# 基于纹理特征的工业缺陷检测系统 — 实验 Notebook

人工智能综合实践课程设计 · MVTec AD metal_nut · 无监督异常检测

**实验内容**：
1. 实验 1：PatchCore / PaDiM / CutPaste 三模型横向对比（按准确率筛选后的最终模型集合）
2. 实验 2：PatchCore 单层 vs 多层 VGG 特征消融（含 k 值网格搜索）
3. 实验 3：数据增强消融（CutPaste）
4. 实验 4：三模型基线实验（朴素基线 vs 最终配置）
5. 实验 5：PatchCore 主方案优化搜索（k × 特征层，全量记忆库）

**运行前请确认**：已按 README 完成环境安装与数据解压，并从项目根目录启动 jupyter。
也可以不逐格运行本 Notebook，直接 `%run run_all_experiments.py` 一键跑完实验 1~4（实验 5 为
主方案配置搜索，单独运行 `python experiments/exp5_patchcore_optimize.py`）。

## 0. 环境准备

In [ ]:
import os
import sys
from pathlib import Path

# 必须先于 numpy/torch 导入设置（Windows：防 OpenBLAS 内存报错、GBK 编码报错）
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")


def _find_project_root() -> Path:
    """定位项目根目录（包含 src/ 与 configs/ 的 defect_detection 目录）。"""
    # 方式 1：由 Notebook 文件所在位置向上找（VS Code / JupyterLab 均适用，不依赖启动目录）
    for var in ("__vsc_ipynb_file__", "__file__"):
        p = globals().get(var)
        if p and str(p).endswith(".ipynb"):
            for parent in Path(p).resolve().parents:
                if (parent / "src").exists() and (parent / "configs").exists():
                    return parent
    # 方式 2：从当前工作目录向上找（经典 Jupyter Notebook 场景）
    d = Path.cwd().resolve()
    for parent in (d, *d.parents):
        if (parent / "src").exists() and (parent / "configs").exists():
            return parent
    raise FileNotFoundError(
        "未找到项目根目录（defect_detection，含 src/ 与 configs/）。\n"
        "请确认 Notebook 位于项目的 experiments/ 目录下，或从项目根目录启动 Jupyter。"
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
print("项目根目录:", PROJECT_ROOT)
print("CUDA 可用:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. 数据准备与校验

In [ ]:
import os
from collections import Counter
import pandas as pd

from src.data.extract_mvtec import extract_metal_nut
from src.utils import set_seed

set_seed(42)
root = extract_metal_nut()  # 幂等：已解压则跳过

# 统计各子集图片数量
counter = Counter()
for p in (root / "metal_nut").rglob("*.png"):
    parts = p.relative_to(root / "metal_nut").parts
    key = f"{parts[0]}/{parts[1]}" if len(parts) >= 3 else str(p.relative_to(root / 'metal_nut').parent)
    counter[key] += 1
display(pd.DataFrame(sorted(counter.items()), columns=["子集", "图片数"]))

## 2. 数据可视化（正常样本 vs 缺陷样本）

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from src.data.dataset import MVTecDataset
from src.utils import denormalize

ds = MVTecDataset(root, split="test", image_size=256, use_augmentation=False)
print("测试集大小:", len(ds))

def show(idx):
    item = ds[idx]
    img = denormalize(item["image"].numpy().transpose(1, 2, 0))
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img); axes[0].set_title(f"图像 ({Path(item['path']).name})")
    if item["label"] == 1:
        axes[1].imshow(item["mask"][0], cmap="gray"); axes[1].set_title("真实缺陷掩码")
    else:
        axes[1].axis("off"); axes[1].set_title("正常样本（无掩码）")
    for ax in axes: ax.axis("off")
    plt.show()

show(0)           # 正常样本
show(len(ds) - 1)  # 缺陷样本（scratch）

## 实验 1：3 种模型横向对比

指标：图像级 AUROC、PixelAUROC、单张平均推理耗时。
输出：`results/comparison.csv`、`results/comparison.md`、`results/heatmaps/` 热力图样例。

> 初版实验实现 5 种技术路线后按测试集准确率筛选，保留前 3 名（PaDiM / PatchCore / CutPaste）；
> 训练过的模型权重缓存在 `checkpoints/`，重跑自动跳过训练；`retrain=True` 强制重训。

In [ ]:
from experiments.exp1_model_compare import run_experiment as exp1

df1 = exp1(retrain=False)
df1

## 实验 2：PatchCore 消融实验

- 单层 VGG 特征 vs 多层特征融合（指标 PixelAUROC）
- 近邻数量 k ∈ {1, 3, 5, 9} 网格搜索

输出：`results/exp2_patchcore_ablation.csv`

In [ ]:
from experiments.exp2_patchcore_ablation import run_experiment as exp2

df2 = exp2(retrain=False)
df2

## 实验 3：数据增强消融实验

CutPaste 在「有 / 无数据增强」下的性能对比（指标 AUROC）。
增强策略：随机水平翻转 + ±5° 旋转 + 亮度扰动（仅训练集）。

输出：`results/exp3_augmentation.csv`

In [ ]:
from experiments.exp3_augmentation import run_experiment as exp3

df3 = exp3(retrain=False)
df3

## 实验 4：三模型基线实验

将每个模型还原为「朴素基线形态」重新评估，与最终配置对比，检验各优化设计是否真实带来收益：

| 模型 | 基线配置 | 最终配置 |
|---|---|---|
| PatchCore | 单层 conv4_3 + k=1 + 全量记忆库（无 coreset 压缩） | 多层融合 + k=1 + 全量记忆库 |
| PaDiM | ResNet18 + layer1~3 + 100 维（论文默认） | VGG16 双层 + 550 维 |
| CutPaste | 无数据增强 | 有数据增强 |

输出：`results/exp4_baseline.csv`、`results/baseline_vs_optimized.csv`、`results/heatmaps_baseline/` 基线热力图。

In [ ]:
from experiments.exp4_baseline import run_experiment as exp4

df4 = exp4(retrain=False)
df4.head(10)

# 基线 vs 优化 汇总对比表（Δ = 优化 − 基线）
pd.read_csv(PROJECT_ROOT / "results" / "baseline_vs_optimized.csv")

## 实验 5：PatchCore 主方案优化搜索

在实验 4 已验证方向（全量记忆库、无增强）上搜索 k ∈ {1,3,5} × 特征层（单层 conv4_3 / 多层融合）共 6 组配置，确定主方案最终配置：多层融合 + k=1 + 全量记忆库 → 0.9477 / 0.9240。

输出：`results/exp5_patchcore_optimize.csv`

In [ ]:
from experiments.exp5_patchcore_optimize import run_experiment as exp5

df5 = exp5(retrain=False)
df5

## 结果汇总

所有实验指标已保存到 `results/` 目录（CSV / Markdown 双格式，可直接贴入报告）。

In [ ]:
for f in sorted((PROJECT_ROOT / "results").glob("*.csv")):
    print("=" * 60)
    print(f.name)
    display(pd.read_csv(f))

## 附：Gradio 演示系统

另开终端运行 `python app.py`，浏览器打开 http://127.0.0.1:7860 即可体验三模型（PatchCore / PaDiM / CutPaste）交互式缺陷检测。